# A 50-Agent Financial Swarm: From Distributed Judgment to a Governed Equity Portfolio

## Introduction

A large swarm is easiest to understand if we stop imagining fifty copies of the same chatbot and instead think of a temporary financial institution. A real investment organization does not ask one person to look at a market, form a view, size every position, challenge every assumption, monitor every risk, and approve the final portfolio. It distributes cognition. Analysts specialize by sector and signal. Risk officers look for failure modes. Portfolio managers care about diversification and implementation. Skeptics challenge consensus. An investment committee converts many incomplete views into one governed decision. A swarm of AI agents can reproduce this organizational principle in software, while making every intermediate judgment explicit enough to inspect.

This notebook builds a **50-agent swarm** around a concrete finance problem: selecting and sizing a long-only U.S. equity portfolio from a liquid universe. The objective is deliberately richer than “ask fifty models for stock picks.” We first construct a numerical market packet from price histories: recent returns, volatility, drawdown, momentum, downside risk, and correlations. We then create fifty Claude Sonnet 5 agents with differentiated mandates. Some agents emphasize momentum, some defensive characteristics, some diversification, some drawdown control, some cross-sectional ranking, and some adversarial skepticism. Each receives the same bounded data environment but a different role, so diversity comes from *mandate heterogeneity* rather than from pretending that identical prompts constitute collective intelligence.

The example uses the `ANTHROPIC_API_KEY` stored in the **Secrets** section of Google Colab. The key is retrieved with `google.colab.userdata`; it is never printed and should never be embedded in the notebook. The model name is centralized in one variable, `MODEL`, so that if Anthropic exposes Claude Sonnet 5 under a versioned API identifier, only that line needs to be changed. The swarm is executed concurrently, because fifty sequential calls would make the architecture unnecessarily slow. A concurrency limit prevents the notebook from launching all calls simultaneously and makes the design easier to adapt to API rate limits.

The architecture has four conceptual layers. The first is **evidence construction**. Financial data are downloaded and transformed into a compact, machine-readable state of the market. The second is **distributed judgment**. Fifty agents receive that state and independently produce structured assessments. The third is **collective aggregation**. Individual opinions are not simply averaged. We examine agreement, disagreement, confidence, and the breadth of support for each asset. Extreme judgments are winsorized and confidence is bounded. This is important because a swarm becomes useful only when its aggregation mechanism is more disciplined than a noisy vote. The fourth layer is **portfolio governance**. The collective signal is converted into weights subject to diversification, position-size, and risk constraints. A final audit cell then asks whether the swarm actually added useful structure relative to simple baselines.

This distinction between *reasoning* and *execution* is central. Claude does not receive authority to trade. The language model produces judgments inside a constrained schema. Deterministic Python performs the portfolio mathematics. This separation is a practical governance pattern for financial AI: use probabilistic models where interpretation and synthesis are valuable; use deterministic code where accounting identities, constraints, and reproducibility matter. The swarm therefore proposes; the portfolio engine disposes.

Why fifty agents? There is nothing magical about the number. It is large enough to make several phenomena visible. First, specialization becomes meaningful: we can allocate multiple agents to the same analytical family and observe whether they agree. Second, disagreement becomes a measurable object rather than an anecdote. Third, redundancy can create robustness: if one call fails or one agent produces an eccentric answer, the system need not collapse. Fourth, the computational economics become explicit. A large swarm consumes tokens and latency, so the notebook records usage and makes it possible to ask whether the marginal agent contributes enough information to justify its cost.

The notebook also illustrates an important limitation. Fifty agents using the same foundation model are **not fifty independent minds**. They share training, architecture, and many latent biases. Prompt diversity creates functional heterogeneity, not statistical independence. Consequently, a swarm should not be treated as a machine for manufacturing certainty. Its greatest value is often the opposite: it externalizes competing hypotheses, quantifies dispersion, and gives the supervisor a richer map of uncertainty.

The financial example is educational rather than an investment recommendation. Historical prices are incomplete descriptions of firms, and the signals used here are intentionally compact so that the architecture remains legible. A production system would add fundamentals, estimates, news, liquidity, transaction costs, factor exposures, corporate actions, mandate restrictions, compliance rules, and independent model validation. It would also separate research data from execution systems and place human approval at consequential checkpoints.

As you work through the ten code cells, focus less on the particular portfolio and more on the institutional design. The interesting object is not any single agent. It is the **coordination mechanism**: how fifty bounded specialists can turn common evidence into differentiated judgments, how those judgments can be aggregated without erasing disagreement, and how a deterministic governance layer can transform collective intelligence into an auditable financial decision. That is the essence of a large swarm.


## Cell 1 — Environment, secrets, and reproducibility

The first cell establishes the operating environment. A swarm is a distributed computational system, so apparently mundane setup choices become part of the research design. We install the Anthropic SDK for Claude, `yfinance` for market data, and the numerical libraries used later for portfolio construction. We then retrieve `ANTHROPIC_API_KEY` from Colab Secrets through `google.colab.userdata`. The notebook never prints the secret, writes it to disk, or places it inside a prompt. This is a small but important example of separating credentials from research artifacts.

The model identifier is defined once as `MODEL = "claude-sonnet-5"`. Anthropic model identifiers can be versioned; if the API exposes Sonnet 5 under a more specific identifier, change only this variable. We also set a random seed for deterministic numerical operations. Language-model outputs themselves can still vary, but the market-data transformations and portfolio calculations remain reproducible conditional on the downloaded observations.

Concurrency is introduced here through `ThreadPoolExecutor`, but the actual swarm is not launched yet. `MAX_WORKERS` deliberately limits simultaneous requests. Fifty agents do not require fifty simultaneous network calls; controlled parallelism is usually better because it respects rate limits, reduces transient failures, and makes the system easier to operate. We also initialize the Anthropic client once and reuse it.

Finally, the cell defines a compact universe of liquid U.S. equities. The purpose is not to claim that these securities form an optimal investment universe. They simply provide enough cross-sectional diversity for the swarm to confront a realistic allocation problem. Later cells will convert their price histories into a bounded evidence packet. The key design principle is already visible: the swarm will not browse freely or invent its own data. All agents reason from a controlled information set prepared by deterministic code.


In [1]:
# CELL 1 — Environment, Colab Secret, and model configuration
!pip -q install anthropic yfinance

import json, re, time, math, random
import numpy as np
import pandas as pd
import yfinance as yf

from google.colab import userdata
from anthropic import Anthropic
from concurrent.futures import ThreadPoolExecutor, as_completed
from scipy.optimize import minimize
from scipy.stats import spearmanr

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

API_KEY = userdata.get("ANTHROPIC_API_KEY")
assert API_KEY, "Add ANTHROPIC_API_KEY to Colab > Secrets and enable notebook access."

# If Anthropic exposes Sonnet 5 under a versioned API identifier, change only this line.
MODEL = "claude-sonnet-5"
MAX_WORKERS = 8
MAX_TOKENS = 2200

client = Anthropic(api_key=API_KEY)

TICKERS = [
    "AAPL","MSFT","NVDA","AMZN","GOOGL","META","AVGO","TSLA","JPM","V",
    "MA","UNH","XOM","COST","HD","PG","JNJ","ABBV","KO","PEP",
    "MRK","CVX","WMT","BAC","CRM","NFLX","AMD","ORCL","CSCO","IBM"
]
print(f"Configured {MODEL} with {len(TICKERS)} securities. API key loaded securely.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 3.5 MB/s eta 0:00:00
Configured claude-sonnet-5 with 30 securities. API key loaded securely.


## Cell 2 — Build the common market evidence

A swarm needs a shared world before it can sustain meaningful disagreement. This cell downloads adjusted daily prices for the predefined equity universe and creates the raw numerical substrate from which every agent will reason. We request several years of history so that short-term momentum can be interpreted against a broader volatility and drawdown background. The exact ending date is left to the data provider, making the notebook reusable whenever it is run.

The important architectural choice is that the language model does **not** fetch prices itself. Market data acquisition is delegated to a conventional data library, and the resulting panel is cleaned in Python. This avoids a common failure mode in financial LLM experiments: asking a model both to discover facts and to reason about them, then losing the ability to distinguish data error from analytical error. Here the provenance boundary is clear. `yfinance` supplies the historical series; Python checks missingness and computes returns; Claude receives only derived evidence.

The cell also prints the date range and dimensions rather than dumping the entire data frame. A fifty-agent system can become expensive if every prompt contains unnecessary observations. We therefore keep the full time series locally for quantitative calculations but later send each agent a compressed feature table. This is analogous to an investment bank or asset manager maintaining a detailed data warehouse while giving decision makers concise analytical packets.

If a ticker has insufficient observations, the cell drops it rather than filling large gaps with invented values. That conservative behavior matters because swarm scale can amplify upstream mistakes: fifty agents reasoning about a corrupted input do not create fifty independent corrections. They create fifty elaborations of the same error. Good swarm design therefore begins with disciplined evidence engineering, not with more agents.


In [2]:
# CELL 2 — Download and clean common market evidence
raw = yf.download(
    TICKERS,
    period="3y",
    interval="1d",
    auto_adjust=True,
    progress=False,
    group_by="column",
    threads=True
)

if isinstance(raw.columns, pd.MultiIndex):
    prices = raw["Close"].copy()
else:
    prices = raw[["Close"]].rename(columns={"Close": TICKERS[0]})

prices = prices.dropna(axis=1, thresh=int(len(prices) * 0.90)).ffill().dropna()
returns = prices.pct_change().dropna()

print("Price history:", prices.index.min().date(), "to", prices.index.max().date())
print("Observations:", len(prices), "| Securities retained:", prices.shape[1])
display(prices.tail())


Price history: 2023-09-22 to 2026-09-21
Observations: 751 | Securities retained: 30


Ticker,AAPL,ABBV,AMD,AMZN,AVGO,BAC,COST,CRM,CSCO,CVX,...,NFLX,NVDA,ORCL,PEP,PG,TSLA,UNH,V,WMT,XOM
Date,,,,,,,,,,,,,,,,,,,,,
2026-09-15,331.339996,263.040009,504.200012,248.419998,338.653320,59.520000,901.349976,255.201019,110.070000,217.770004,...,77.900002,212.169998,140.350006,135.500000,146.669998,356.579987,375.929993,375.619995,108.089996,169.320007
2026-09-16,332.410004,262.510010,512.500000,245.960007,338.892914,57.900002,893.739990,250.099991,107.739998,211.539993,...,76.410004,213.899994,143.160004,134.339996,147.020004,358.079987,375.260010,370.929993,107.500000,163.320007
2026-09-17,337.000000,264.019989,545.090027,251.190002,346.668732,58.180000,893.929993,242.850006,110.239998,211.570007,...,75.309998,219.339996,150.589996,133.660004,147.550003,366.200012,375.209991,369.929993,106.790001,163.270004
2026-09-18,336.130005,263.959991,559.820007,253.710007,356.959991,57.730000,895.309998,237.919998,109.510002,209.509995,...,71.790001,222.270004,147.610001,129.750000,146.389999,364.269989,376.899994,368.290009,106.730003,163.539993
2026-09-21,338.980011,264.480011,615.520020,258.450012,362.660004,57.959999,898.479980,236.419998,111.459999,203.669998,...,73.360001,227.380005,148.559998,129.589996,146.080002,375.299988,377.559998,369.950012,107.440002,158.300003


## Cell 3 — Transform prices into an analytical state

The third cell converts raw prices into a cross-sectional state vector. Each security receives a small set of interpretable features: one-month, three-month, six-month, and twelve-month returns; annualized volatility; maximum drawdown; downside volatility; and a simple risk-adjusted momentum measure. We also retain the return matrix and correlation matrix for the deterministic portfolio layer.

These variables are intentionally familiar. The goal of the notebook is to expose swarm mechanics, not to hide them behind an elaborate proprietary factor model. Yet even simple features create genuine tensions among agents. A momentum specialist may favor an asset with strong twelve-month performance, while a defensive specialist may penalize the same security for high volatility or a severe drawdown. A diversification agent may value a moderately attractive asset because its correlation pattern improves the portfolio. Those conflicts are exactly what make a swarm informative.

We standardize selected features cross-sectionally with z-scores. Standardization does not tell the agents what to conclude; it puts different metrics onto comparable scales and allows the later aggregation layer to combine quantitative evidence with model judgments. We protect against zero standard deviations and infinite values so that one pathological feature cannot contaminate the packet.

At the end, the cell constructs `market_packet`, a list of dictionaries rounded to a sensible precision. This object is what Claude will see. The compression is deliberate: rather than sending thousands of daily prices to fifty agents, we send a concise state representation. In production, this state could be far richer—fundamentals, analyst revisions, options-implied information, news embeddings, liquidity measures, macro exposures—but the architectural rule would remain the same. The evidence packet should be explicit, bounded, auditable, and small enough that every agent can reason over the same information without unnecessary token expenditure.


In [3]:
# CELL 3 — Engineer a compact, auditable market state
def trailing_return(s, days):
    if len(s) <= days:
        return np.nan
    return s.iloc[-1] / s.iloc[-days-1] - 1

def max_drawdown(s):
    wealth = s / s.iloc[0]
    return (wealth / wealth.cummax() - 1).min()

rows = []
for ticker in prices.columns:
    p = prices[ticker].dropna()
    r = p.pct_change().dropna()
    downside = r[r < 0]
    rows.append({
        "ticker": ticker,
        "ret_1m": trailing_return(p, 21),
        "ret_3m": trailing_return(p, 63),
        "ret_6m": trailing_return(p, 126),
        "ret_12m": trailing_return(p, 252),
        "vol_ann": r.std() * np.sqrt(252),
        "downside_vol": downside.std() * np.sqrt(252) if len(downside) else 0.0,
        "max_drawdown_3y": max_drawdown(p),
    })

features = pd.DataFrame(rows).set_index("ticker")
features["risk_adj_mom"] = features["ret_12m"] / features["vol_ann"].replace(0, np.nan)

z_cols = ["ret_3m","ret_6m","ret_12m","vol_ann","downside_vol","max_drawdown_3y","risk_adj_mom"]
for c in z_cols:
    sd = features[c].std()
    features[c + "_z"] = (features[c] - features[c].mean()) / (sd if sd > 0 else 1.0)

features = features.replace([np.inf, -np.inf], np.nan).fillna(0.0)
cov_ann = returns[features.index].cov() * 252
corr = returns[features.index].corr()

market_packet = (
    features.reset_index()
    .round(4)
    .to_dict(orient="records")
)

display(features.round(3))
print("Compact packet rows sent to each agent:", len(market_packet))


,ret_1m,ret_3m,ret_6m,ret_12m,vol_ann,downside_vol,max_drawdown_3y,risk_adj_mom,ret_3m_z,ret_6m_z,ret_12m_z,vol_ann_z,downside_vol_z,max_drawdown_3y_z,risk_adj_mom_z
ticker,,,,,,,,,,,,,,,
AAPL,0.089,0.142,0.369,0.430,0.268,0.193,-0.334,1.603,0.386,0.409,0.355,-0.306,-0.272,-0.010,0.623
ABBV,0.010,0.158,0.310,0.228,0.247,0.194,-0.207,0.923,0.483,0.255,0.010,-0.481,-0.257,0.834,0.138
AMD,0.311,0.116,2.057,2.898,0.587,0.359,-0.630,4.933,0.223,4.764,4.562,2.361,1.862,-1.991,2.997
AMZN,-0.006,0.110,0.258,0.118,0.322,0.204,-0.309,0.365,0.189,0.123,-0.178,0.145,-0.136,0.156,-0.259
AVGO,-0.002,-0.073,0.174,0.060,0.500,0.333,-0.411,0.119,-0.942,-0.094,-0.277,1.633,1.525,-0.530,-0.435
BAC,-0.058,0.015,0.242,0.136,0.247,0.184,-0.275,0.550,-0.395,0.080,-0.148,-0.488,-0.387,0.381,-0.128
COST,-0.038,-0.054,-0.073,-0.051,0.204,0.149,-0.207,-0.251,-0.822,-0.733,-0.466,-0.846,-0.847,0.834,-0.698
CRM,0.153,0.578,0.218,-0.024,0.388,0.272,-0.587,-0.062,3.066,0.019,-0.420,0.696,0.739,-1.701,-0.564
CSCO,0.017,-0.079,0.449,0.656,0.264,0.211,-0.175,2.491,-0.978,0.613,0.740,-0.346,-0.040,1.053,1.256


Compact packet rows sent to each agent: 30


## Cell 4 — Create fifty differentiated agents

This cell is where the “swarm” becomes more than a metaphor. We construct exactly fifty agent specifications. The agents are divided across ten analytical families, with five agents in each family. The families include momentum, reversal, defensive quality proxies, drawdown control, volatility efficiency, diversification, cross-sectional ranking, regime sensitivity, skeptical review, and balanced portfolio judgment. Because the available dataset is price-based, mandates are carefully phrased so agents do not pretend to possess fundamental information that they have not been given.

Within each family, five variants receive slightly different emphases. One may privilege robustness, another dispersion, another downside asymmetry, another ranking stability, and another explicit falsification. This creates controlled diversity while preserving a recognizable organizational structure. We assign every agent a unique ID such as `momentum_01`. That ID will travel with the response all the way through aggregation, allowing us to trace any final score back to the contributing specialist.

A crucial point is that specialization is encoded in the **mandate**, not by giving different agents secret facts. All agents see the same evidence packet. This lets us attribute differences primarily to analytical perspective. In a more advanced swarm, information itself could also be partitioned—for example, sector agents might receive industry documents while macro agents receive rates and inflation data—but doing both at once would make the pedagogical experiment harder to interpret.

The roster is represented as ordinary Python data. There is no hidden orchestration framework. This makes the notebook useful for teaching because students can inspect the entire institutional design in one object. The cell ends by verifying that there are fifty unique IDs and displaying the family counts. Before spending a single API token, we therefore know the intended cognitive composition of the swarm. This is governance by construction: define who is allowed to reason, from what perspective, and under what identity before asking the model to do any work.


In [4]:
# CELL 4 — Define 50 specialists: 10 families × 5 variants
families = {
    "momentum": "Emphasize persistence across 3m, 6m and 12m returns; penalize fragile one-window strength.",
    "reversal": "Look for overextended moves and plausible mean-reversion opportunities using only supplied price statistics.",
    "defensive": "Prefer lower volatility, lower downside volatility and shallower drawdowns while retaining acceptable returns.",
    "drawdown": "Act as a capital-preservation specialist; treat severe historical drawdown as a major warning.",
    "efficiency": "Emphasize return earned per unit of volatility and downside risk.",
    "diversification": "Favor candidates that can contribute to a diversified portfolio; avoid simply chasing the strongest return.",
    "cross_section": "Rank securities relative to peers using the full feature vector and internal consistency across horizons.",
    "regime": "Ask which signals appear robust if the recent return regime changes; penalize dependence on one horizon.",
    "skeptic": "Act as a red-team analyst. Search for reasons the apparent leaders may be misleading or unstable.",
    "balanced": "Integrate return, volatility, downside, drawdown and robustness without allowing one metric to dominate."
}

variants = [
    "Prioritize robustness and consistency.",
    "Pay special attention to cross-sectional dispersion.",
    "Emphasize downside asymmetry and failure modes.",
    "Prefer stable rankings across multiple horizons.",
    "Try to falsify the obvious conclusion before scoring."
]

agents = []
for family, mandate in families.items():
    for i, variant in enumerate(variants, start=1):
        agents.append({
            "agent_id": f"{family}_{i:02d}",
            "family": family,
            "mandate": mandate,
            "variant": variant
        })

assert len(agents) == 50
assert len({a["agent_id"] for a in agents}) == 50

print("Total agents:", len(agents))
display(pd.Series([a["family"] for a in agents]).value_counts().rename("agents").to_frame())


Total agents: 50


,agents
momentum,5
reversal,5
defensive,5
drawdown,5
efficiency,5
diversification,5
cross_section,5
regime,5
skeptic,5
balanced,5


## Cell 5 — Define the agent contract and structured output

Large swarms fail quickly if every agent is allowed to answer in an arbitrary essay. This cell defines the contract that converts Claude from a conversational model into a bounded analytical worker. Each agent receives its mandate, the common market packet, and explicit instructions to return JSON with a score for every ticker, confidence, a concise rationale, and a list of portfolio-level risks. Scores are constrained to a symmetric scale from -5 to +5, where positive values indicate relative attractiveness within the supplied universe rather than an absolute forecast.

The prompt explicitly prohibits invented data. Agents are told to use only the supplied packet and to acknowledge when their mandate cannot infer something from the evidence. This matters particularly in finance, where a fluent model can otherwise slide from price behavior into unsupported claims about earnings, management, or valuation. The schema keeps the epistemic boundary visible.

We also request a portfolio-level thesis and risks. These qualitative fields are not used directly in the mathematical optimizer, but they are valuable for audit and later inspection. A swarm should preserve explanations rather than collapsing immediately into a scalar vote. The final notebook cell will sample these narratives so that we can compare numerical consensus with the arguments that produced it.

The helper function makes one Claude call and returns both the parsed text and usage metadata. Token usage is recorded because swarm economics are part of the experiment. Fifty-agent architectures can be powerful, but scale is not free. By measuring input and output tokens, we can later estimate whether a smaller swarm might capture most of the same signal. The cell does not execute the swarm; it defines the protocol. Separating protocol definition from execution makes debugging easier and prevents accidental repeated API expenditure while editing prompts.


In [14]:
# CELL 5 — Native structured output contract for Claude Sonnet 5

SYSTEM = """
You are one specialist inside a governed 50-agent financial research swarm.

Use ONLY the numerical evidence supplied by the orchestrator.
Do not invent fundamentals, news, valuation data, forecasts, or events.

Your role is to provide a bounded analytical judgment.
You do NOT construct or execute the final portfolio.
"""

AGENT_SCHEMA = {
    "type": "object",
    "properties": {

        "agent_id": {
            "type": "string"
        },

        "family": {
            "type": "string"
        },

        "portfolio_thesis": {
            "type": "string"
        },

        "portfolio_risks": {
            "type": "array",
            "items": {"type": "string"}
        },

        "assessments": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {

                    "ticker": {
                        "type": "string"
                    },

                    "score": {
                        "type": "number"
                    },

                    "confidence": {
                        "type": "number"
                    },

                    "rationale": {
                        "type": "string"
                    }
                },

                "required": [
                    "ticker",
                    "score",
                    "confidence",
                    "rationale"
                ],

                "additionalProperties": False
            }
        }
    },

    "required": [
        "agent_id",
        "family",
        "portfolio_thesis",
        "portfolio_risks",
        "assessments"
    ],

    "additionalProperties": False
}


def build_prompt(agent, packet):

    return f"""
You are agent {agent['agent_id']}.

SPECIALIST FAMILY:
{agent['family']}

MANDATE:
{agent['mandate']}

ANALYTICAL VARIANT:
{agent['variant']}

You must assess EVERY security contained in MARKET_PACKET.

SCORING:

+5 = exceptionally attractive relative to this universe
+3 = clearly attractive
+1 = mildly attractive
 0 = neutral
-1 = mildly unattractive
-3 = clearly unattractive
-5 = exceptionally unattractive

Confidence must be between 0 and 1.

Keep each individual rationale concise.
Use only the evidence supplied below.

MARKET_PACKET:

{json.dumps(packet)}
"""


def call_agent(agent):

    msg = client.messages.create(

        model=MODEL,

        # Sonnet 5 needs adequate room for adaptive thinking
        # plus the complete structured response.
        max_tokens=8000,

        # Medium effort is sufficient for this bounded
        # cross-sectional analytical task.
        output_config={
            "effort": "medium",
            "format": {
                "type": "json_schema",
                "schema": AGENT_SCHEMA
            }
        },

        system=SYSTEM,

        messages=[
            {
                "role": "user",
                "content": build_prompt(
                    agent,
                    market_packet
                )
            }
        ]
    )

    # Protect explicitly against truncation.
    if msg.stop_reason != "end_turn":
        raise RuntimeError(
            f"Agent {agent['agent_id']} stopped with "
            f"stop_reason={msg.stop_reason}"
        )

    text = next(
        block.text
        for block in msg.content
        if block.type == "text"
    )

    return {
        "agent_id": agent["agent_id"],
        "family": agent["family"],
        "raw_text": text,
        "stop_reason": msg.stop_reason,
        "input_tokens": getattr(
            msg.usage,
            "input_tokens",
            0
        ),
        "output_tokens": getattr(
            msg.usage,
            "output_tokens",
            0
        )
    }


print(
    "Claude Sonnet 5 structured-output "
    "contract ready."
)

Claude Sonnet 5 structured-output contract ready.


## Cell 6 — Launch the 50-agent swarm concurrently

Now the distributed institution goes to work. The sixth cell submits one task for each of the fifty agent specifications using a bounded thread pool. Every task calls the same Claude Sonnet 5 model, but each call carries a different mandate and agent identity. The market evidence remains common. This produces a useful experimental structure: one foundation model, one market state, fifty functional perspectives.

The concurrency mechanism is intentionally straightforward. `ThreadPoolExecutor` is sufficient because the dominant latency is network I/O rather than local computation. `MAX_WORKERS` controls how many requests can be in flight at once. If the API account has tighter rate limits, reduce that number. If a request fails, the exception is captured and stored against the agent ID rather than terminating the entire experiment. This is an important swarm property: local failure should degrade the system gracefully instead of becoming system failure.

A progress message is printed as each agent finishes. The results are stored in a dictionary keyed by agent ID, preserving provenance regardless of completion order. We also aggregate token counts after execution. This makes the cost of collective cognition visible. In a production architecture, one could add retries with exponential backoff, per-agent budgets, caching, and early stopping when marginal opinions cease to change the consensus materially.

Notice what this cell does *not* do. Agents do not communicate with one another. This is a first-round “independent committee” design, which is useful because it avoids herding. Later architectures could add deliberation rounds in which agents read anonymized peer arguments and revise their views. But independence in the first pass gives us a clean measure of spontaneous agreement and disagreement. Once this cell completes, the notebook contains fifty bounded judgments over the same financial state—enough to study collective behavior rather than merely model behavior.


In [15]:
# CELL 6 — Execute all 50 Claude agents with bounded concurrency
results = {}
errors = {}

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(call_agent, a): a for a in agents}
    for n, future in enumerate(as_completed(futures), start=1):
        agent = futures[future]
        try:
            result = future.result()
            results[agent["agent_id"]] = result
            print(f"[{n:02d}/50] completed {agent['agent_id']}")
        except Exception as e:
            errors[agent["agent_id"]] = repr(e)
            print(f"[{n:02d}/50] FAILED {agent['agent_id']}: {e}")

usage = pd.DataFrame(results.values())[["agent_id","family","input_tokens","output_tokens"]]
print("\nSuccessful:", len(results), "| Failed:", len(errors))
print("Total input tokens:", int(usage["input_tokens"].sum()) if len(usage) else 0)
print("Total output tokens:", int(usage["output_tokens"].sum()) if len(usage) else 0)
display(usage.groupby("family")[["input_tokens","output_tokens"]].sum())


[01/50] completed reversal_02
[02/50] completed momentum_03
[03/50] completed momentum_01
[04/50] completed reversal_01
[05/50] completed reversal_03
[06/50] completed momentum_04
[07/50] completed momentum_02
[08/50] completed momentum_05
[09/50] completed defensive_01
[10/50] completed reversal_04
[11/50] completed defensive_02
[12/50] completed defensive_03
[13/50] completed defensive_04
[14/50] completed reversal_05
[15/50] completed defensive_05
[16/50] completed drawdown_01
[17/50] completed drawdown_04
[18/50] completed drawdown_02
[19/50] completed drawdown_03
[20/50] completed efficiency_01
[21/50] completed efficiency_02
[22/50] completed efficiency_04
[23/50] completed efficiency_03
[24/50] completed diversification_01
[25/50] completed diversification_02
[26/50] completed diversification_04
[27/50] completed drawdown_05
[28/50] completed diversification_03
[29/50] completed efficiency_05
[30/50] completed cross_section_02
[31/50] completed diversification_05
[32/50] complet

,input_tokens,output_tokens
family,,
balanced,39390,11064
cross_section,39375,14373
defensive,39410,10327
diversification,39400,11291
drawdown,39385,13090
efficiency,39340,13134
momentum,39400,13742
regime,39375,13203
reversal,39385,11077


## Cell 7 — Parse, validate, and normalize the swarm

Language models are probabilistic components, so their outputs must be treated as untrusted inputs to downstream software. The seventh cell is therefore a validation layer. It extracts JSON, verifies that the expected fields exist, clips scores and confidence to their allowed ranges, and records malformed responses separately. A financial system should never assume that because a model was instructed to return a schema, the schema will always be obeyed.

For every valid response, we expand the agent’s ticker-level assessments into a tidy table with one row per agent-security pair. This format makes the swarm analyzable using ordinary statistical tools. We attach the agent family, confidence, rationale, and identifier to every observation. Missing ticker assessments remain visible rather than being silently imputed. The resulting table is effectively the swarm’s “opinion tape.”

The validation step also limits the influence of extreme values. Scores are clipped to the contractual range, and confidence is bounded between zero and one. Later aggregation will use robust statistics, but defensive validation should happen before any arithmetic. This is analogous to validating trade messages before they reach an order management system.

The cell reports the number of valid and invalid agents and shows a small sample. If many agents fail parsing, the correct response is not to proceed confidently with fewer opinions; it is to inspect the prompt or API responses. A swarm can tolerate occasional node failure, but systematic failure indicates an architectural problem. By making this diagnostic explicit, the notebook teaches a broader lesson: the sophistication of an AI system is not measured only by the intelligence of its model. Reliability comes from contracts, validation, provenance, and graceful handling of imperfect components.


In [17]:
# CELL 7 — Parse and validate every opinion before aggregation
def extract_json(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.I | re.S)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start, end = text.find("{"), text.rfind("}")
        if start >= 0 and end > start:
            return json.loads(text[start:end+1])
        raise

rows, parsed, invalid = [], {}, {}

for agent_id, result in results.items():
    try:
        obj = extract_json(result["raw_text"])
        assessments = obj.get("assessments", [])
        if not assessments:
            raise ValueError("No assessments found.")
        parsed[agent_id] = obj
        family = result["family"]

        for a in assessments:
            ticker = str(a["ticker"]).upper()
            if ticker not in features.index:
                continue
            score = float(np.clip(float(a["score"]), -5, 5))
            confidence = float(np.clip(float(a["confidence"]), 0, 1))
            rows.append({
                "agent_id": agent_id,
                "family": family,
                "ticker": ticker,
                "score": score,
                "confidence": confidence,
                "rationale": str(a.get("rationale", ""))[:300]
            })
    except Exception as e:
        invalid[agent_id] = repr(e)

opinions = pd.DataFrame(rows)
print("Valid agents:", len(parsed), "| Invalid responses:", len(invalid))
print("Agent-security opinions:", len(opinions))
display(opinions.head(12))


Valid agents: 50 | Invalid responses: 0
Agent-security opinions: 1500


,agent_id,family,ticker,score,confidence,rationale
0,reversal_02,reversal,AAPL,-1.0,0.40,"Moderate positive z-scores across timeframes, ..."
1,reversal_02,reversal,ABBV,0.0,0.30,"Near-neutral z-scores, no strong reversion sig..."
2,reversal_02,reversal,AMD,-5.0,0.85,Extreme 6m/12m z-scores (4.7/4.6) and highest ...
3,reversal_02,reversal,AMZN,0.0,0.30,"All z-scores near zero, no dispersion signal."
4,reversal_02,reversal,AVGO,1.0,0.35,"Negative 3m return z with elevated vol, mild r..."
5,reversal_02,reversal,BAC,0.0,0.25,"Modest negative z-scores, no strong signal."
6,reversal_02,reversal,COST,1.0,0.40,Broadly negative return z-scores across timefr...
7,reversal_02,reversal,CRM,-3.0,0.60,Extreme 3m z-score (3.07) with deteriorating 1...
8,reversal_02,reversal,CSCO,-1.0,0.35,"Strong 6m/12m momentum but negative 3m z, some..."
9,reversal_02,reversal,CVX,-1.0,0.30,"Positive 3m z with negative 6m z, mixed signal..."


## Cell 8 — Aggregate disagreement into collective conviction

This cell turns fifty individual judgments into a collective signal without pretending that disagreement is noise to be discarded. For each ticker we calculate the median score, mean score, standard deviation, number of observations, average confidence, and the fraction of agents assigning a positive score. The median is especially useful because it is robust to a few eccentric agents. Dispersion becomes an explicit uncertainty measure.

We then define a simple `swarm_conviction` statistic. It rewards a strong median view and high average confidence, while penalizing disagreement. The exact formula is not sacred; it is transparent so students can challenge and replace it. The important idea is architectural: aggregation is itself a model and should be visible, testable, and governed. A majority vote would throw away intensity. A simple mean would be sensitive to outliers. Confidence weighting alone could reward unjustified certainty. The chosen combination makes those trade-offs explicit.

We also calculate family-level means. This allows us to see whether apparent consensus is broad or whether it is driven by one analytical school. For example, a security might score highly among momentum agents but poorly among drawdown specialists. The overall mean could hide that institutional conflict. A family matrix exposes it.

The output is sorted by swarm conviction and becomes the bridge between probabilistic judgment and deterministic portfolio construction. Importantly, the swarm still has not “chosen a portfolio.” It has produced a ranked field of collective assessments with associated uncertainty. That separation matters. An investment committee may have strong views, but portfolio weights must still respect covariance, concentration, and mandate constraints. The next cell performs that conversion mathematically. In this design, collective intelligence generates *preferences*; deterministic optimization translates preferences into *actions*.


In [18]:
# CELL 8 — Robust collective aggregation and disagreement map
if opinions.empty:
    raise RuntimeError("No valid agent opinions were parsed.")

summary = opinions.groupby("ticker").agg(
    median_score=("score","median"),
    mean_score=("score","mean"),
    disagreement=("score","std"),
    avg_confidence=("confidence","mean"),
    positive_share=("score", lambda x: (x > 0).mean()),
    n_agents=("agent_id","nunique")
)

summary["disagreement"] = summary["disagreement"].fillna(0)
summary["swarm_conviction"] = (
    summary["median_score"]
    * summary["avg_confidence"]
    / (1.0 + summary["disagreement"])
)

family_matrix = opinions.pivot_table(
    index="ticker", columns="family", values="score", aggfunc="mean"
)

summary = summary.sort_values("swarm_conviction", ascending=False)

print("Top collective convictions:")
display(summary.round(3).head(15))
print("\nFamily-level mean scores:")
display(family_matrix.loc[summary.index].round(2).head(15))


Top collective convictions:


,median_score,mean_score,disagreement,avg_confidence,positive_share,n_agents,swarm_conviction
ticker,,,,,,,
CSCO,3.0,1.96,1.456,0.605,0.86,50,0.739
JNJ,3.0,3.20,1.948,0.674,0.90,50,0.686
KO,3.0,2.36,1.770,0.627,0.84,50,0.679
AAPL,2.5,1.78,1.345,0.627,0.82,50,0.669
MRK,3.0,2.74,2.174,0.658,0.86,50,0.622
CVX,2.0,1.52,1.199,0.572,0.80,50,0.520
XOM,2.0,1.56,1.327,0.577,0.82,50,0.496
ABBV,1.0,1.48,0.974,0.581,0.86,50,0.294
GOOGL,1.0,0.92,0.804,0.529,0.78,50,0.293



Family-level mean scores:


family,balanced,cross_section,defensive,diversification,drawdown,efficiency,momentum,regime,reversal,skeptic
ticker,,,,,,,,,,
CSCO,3.0,2.6,2.8,2.6,2.8,3.0,0.8,2.2,-1.2,1.0
JNJ,3.6,3.8,5.0,4.0,4.6,5.0,2.4,3.4,-1.0,1.2
KO,2.6,2.6,4.6,2.8,4.6,3.4,1.2,1.8,-0.8,0.8
AAPL,3.0,2.8,1.0,1.2,1.6,3.0,3.0,2.2,-0.6,0.6
MRK,3.6,4.2,3.6,3.8,0.6,4.4,4.0,3.8,-2.0,1.4
CVX,1.8,1.0,2.6,2.4,2.8,2.8,0.2,1.0,-0.2,0.8
XOM,1.8,1.4,3.0,2.4,2.8,2.8,0.2,1.0,-0.8,1.0
ABBV,1.8,1.2,2.6,1.8,2.6,1.2,1.0,1.8,0.0,0.8
GOOGL,1.4,1.0,1.0,1.2,1.2,1.0,0.2,1.8,0.0,0.4


## Cell 9 — Convert swarm conviction into governed portfolio weights

The ninth cell is the portfolio-engineering layer. We select a manageable number of securities with positive swarm conviction and combine that conviction with the historical covariance matrix. The optimizer seeks a portfolio that rewards collective preference while penalizing variance. It is constrained to be long-only, fully invested, and capped at a maximum weight per security. These constraints are intentionally simple but demonstrate how an AI swarm can sit upstream of a conventional quantitative control system.

The optimization objective contains a tunable risk-aversion coefficient. A higher value gives covariance more influence; a lower value allows the swarm’s ranking to dominate. Conviction is normalized before optimization so that its scale is compatible with the risk term. We use SciPy’s SLSQP solver and check whether the optimization succeeds. If it fails, the notebook raises an error rather than silently producing arbitrary weights.

This division of labor is one of the most important lessons in the notebook. Claude is well suited to heterogeneous interpretation and synthesis, but it should not be trusted to “remember” that weights sum to one or that a 12% cap must never be breached. Mathematical constraints belong in deterministic code. The portfolio is therefore the product of two systems: a probabilistic judgment layer and a deterministic governance layer.

We calculate ex-post annualized volatility from the historical covariance matrix and display the resulting weights alongside swarm conviction and disagreement. This makes a subtle point visible: the asset with the highest conviction need not receive the highest weight if it contributes excessive covariance risk. Collective judgment and portfolio construction answer different questions. A mature autonomous financial architecture preserves that distinction instead of allowing an eloquent model response to become an executable allocation directly.


In [20]:
# CELL 9 — Deterministic portfolio construction under explicit constraints
eligible = summary[summary["swarm_conviction"] > 0].head(12).index.tolist()
if len(eligible) < 5:
    eligible = summary.head(min(12, len(summary))).index.tolist()

alpha = summary.loc[eligible, "swarm_conviction"].values.astype(float)
alpha = (alpha - alpha.min()) / (alpha.max() - alpha.min() + 1e-9)
Sigma = cov_ann.loc[eligible, eligible].values

n = len(eligible)
MAX_WEIGHT = 0.15
RISK_AVERSION = 4.0

def objective(w):
    reward = alpha @ w
    variance = w @ Sigma @ w
    return -(reward - RISK_AVERSION * variance)

constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
bounds = [(0.0, MAX_WEIGHT) for _ in range(n)]
x0 = np.repeat(1.0 / n, n)

opt = minimize(objective, x0, method="SLSQP", bounds=bounds, constraints=constraints)
if not opt.success:
    raise RuntimeError(opt.message)

weights = pd.Series(opt.x, index=eligible, name="weight")
portfolio = summary.loc[eligible].copy()
portfolio["weight"] = weights
portfolio = portfolio.sort_values("weight", ascending=False)

port_vol = float(np.sqrt(weights.values @ Sigma @ weights.values))
print(f"Historical covariance-implied annualized volatility: {port_vol:.2%}")
print(f"Max position: {weights.max():.2%} | Sum of weights: {weights.sum():.6f}")
display(portfolio[["weight","swarm_conviction","median_score","disagreement","avg_confidence"]].round(4))


Historical covariance-implied annualized volatility: 12.60%
Max position: 15.00% | Sum of weights: 1.000000


,weight,swarm_conviction,median_score,disagreement,avg_confidence
ticker,,,,,
CSCO,0.15,0.7389,3.0,1.4563,0.6050
MRK,0.15,0.6223,3.0,2.1742,0.6584
AAPL,0.15,0.6690,2.5,1.3445,0.6274
CVX,0.15,0.5198,2.0,1.1993,0.5716
JNJ,0.15,0.6858,3.0,1.9483,0.6740
KO,0.15,0.6795,3.0,1.7700,0.6274
XOM,0.10,0.4955,2.0,1.3273,0.5766
ABBV,0.00,0.2943,1.0,0.9739,0.5810
GOOGL,0.00,0.2932,1.0,0.8041,0.5290


## Cell 10 — Audit the swarm and test whether scale added information

The final code cell treats the swarm itself as an object of research. We compare the full fifty-agent consensus with progressively smaller random subswarms. For each subswarm size, we repeatedly sample agents, recompute ticker-level mean scores, and measure the rank correlation with the full-swarm ranking. This creates a simple convergence curve: if ten agents already reproduce nearly all of the fifty-agent ordering, then the additional forty calls may add little for this problem. If agreement improves materially as the swarm grows, scale may be buying robustness.

We also report family-level disagreement and show representative portfolio-level risks extracted from agent narratives. This combines quantitative and qualitative auditing. A high consensus score should not erase the fact that one specialist family may be systematically dissenting. Conversely, broad agreement across heterogeneous families can be more informative than fifty near-duplicate votes.

This cell is deliberately diagnostic rather than celebratory. Large swarms are not automatically better. They can amplify shared model biases, consume substantial tokens, and create a false impression of independence. Because all fifty agents use the same Claude foundation model, their errors may be correlated even when their mandates differ. The convergence analysis therefore asks an economic question: how much marginal information is being purchased by additional agents?

In a production experiment, this audit would be extended with out-of-sample returns, turnover, transaction costs, benchmark comparisons, factor attribution, stress scenarios, and repeated runs across market regimes. One could also compare heterogeneous-model swarms, where different foundation models or quantitative agents contribute genuinely different error structures. For teaching purposes, however, this final cell closes the loop. We began with raw market data, created fifty bounded specialists, aggregated their judgments, imposed deterministic portfolio controls, and then evaluated the swarm’s own informational efficiency. The system is not merely autonomous; it is inspectable.


In [21]:
# CELL 10 — Audit scale: does a larger swarm converge toward a stable ranking?
rng = np.random.default_rng(SEED)
valid_agents = opinions["agent_id"].unique().tolist()

full_rank = (
    opinions.groupby("ticker")["score"].mean()
    .reindex(features.index)
    .fillna(0)
)

sizes = [5, 10, 20, 30, 40, min(50, len(valid_agents))]
sizes = sorted(set(s for s in sizes if s <= len(valid_agents)))
audit_rows = []

for size in sizes:
    correlations = []
    for _ in range(100):
        chosen = rng.choice(valid_agents, size=size, replace=False)
        sub = opinions[opinions["agent_id"].isin(chosen)]
        sub_rank = sub.groupby("ticker")["score"].mean().reindex(full_rank.index).fillna(0)
        rho = spearmanr(full_rank.values, sub_rank.values).statistic
        if np.isfinite(rho):
            correlations.append(rho)
    audit_rows.append({
        "agents": size,
        "mean_rank_corr_vs_full": np.mean(correlations),
        "p10": np.quantile(correlations, 0.10),
        "p90": np.quantile(correlations, 0.90)
    })

audit = pd.DataFrame(audit_rows)
display(audit.round(3))

family_disagreement = (
    opinions.groupby("family")["score"].std()
    .sort_values(ascending=False)
    .rename("score_std")
)
print("\nWithin-family score dispersion:")
display(family_disagreement.to_frame().round(3))

print("\nRepresentative portfolio-level risks from the swarm:")
for agent_id in list(parsed)[:8]:
    risks = parsed[agent_id].get("portfolio_risks", [])
    print(f"- {agent_id}: {risks}")

print("\nEducational research example only; no orders are generated or transmitted.")


,agents,mean_rank_corr_vs_full,p10,p90
0,5,0.965,0.933,0.989
1,10,0.980,0.961,0.992
2,20,0.990,0.982,0.997
3,30,0.995,0.988,0.999
4,40,0.998,0.995,1.000
5,50,1.000,1.000,1.000



Within-family score dispersion:


,score_std
family,
defensive,2.872
drawdown,2.788
efficiency,2.739
cross_section,2.476
momentum,2.433
balanced,2.270
regime,2.216
diversification,2.094
skeptic,1.636



Representative portfolio-level risks from the swarm:
- reversal_02: ['Momentum persistence could continue longer than mean-reversion signals suggest, especially in AMD/CRM/META', 'Low-confidence reversal calls on defensive/staples names may reflect structural rotation rather than mispricing', "Cross-sectional z-scores don't capture fundamental catalysts, only relative price stretch"]
- momentum_03: ['AMD-like extreme momentum concentration risk of sharp mean-reversion', 'Single-window spikes (e.g., CRM, META, MSFT 3m surges) may reverse once catalyst fades', 'High vol/downside-vol names carry asymmetric crash risk if momentum breaks', 'Sector crowding in mega-cap tech momentum leaves portfolio vulnerable to common factor shock']
- momentum_01: ['Extreme dispersion driven by a few outlier names (e.g., AMD) could concentrate factor exposure', 'High-vol persistence winners may mean-revert sharply, raising drawdown risk', 'Multicollinearity between 3m and 6m/12m returns can overstate true

## Conclusion

This notebook demonstrates why a large agent swarm is more interesting than a large number of API calls. The central object is an **institutional architecture**. Fifty Claude Sonnet 5 agents were organized into specialist families, given a common and bounded market state, required to return structured judgments, and prevented from directly controlling portfolio weights. Their opinions were validated, preserved with provenance, aggregated using robust statistics, and translated into an allocation only through deterministic optimization and explicit constraints.

Several lessons follow. First, scale is useful only when it is accompanied by heterogeneity. Fifty identical prompts would mostly purchase repetition. Here the agents differ by mandate, creating productive tension between momentum, reversal, defensive, diversification, risk-control, skeptical, and balanced perspectives. The disagreement among these specialists is not a defect. It is information about uncertainty and model dependence.

Second, a swarm needs governance more than it needs eloquence. The most important cells are arguably not the ones that call Claude but the ones that define the evidence boundary, validate outputs, aggregate opinions, enforce portfolio constraints, and audit convergence. These layers turn probabilistic language-model behavior into a system that can be inspected and challenged. In finance, that separation is essential: an agent may propose a judgment, but deterministic controls should enforce accounting, risk, mandate, and execution rules.

Third, fifty agents do not create fifty statistically independent sources of truth. They share the same foundation model and can therefore share blind spots. Functional diversity reduces some forms of correlated reasoning but does not eliminate common-model risk. A more advanced architecture could mix language models, quantitative models, retrieval systems, simulation agents, and human specialists. It could also introduce deliberation: a first round of independent opinions, a second round of adversarial critique, and a final revision round before aggregation.

Finally, the notebook makes swarm economics measurable. Token usage, failure rates, disagreement, and subswarm convergence can all be observed. This allows a designer to ask not “How many agents can I deploy?” but “How many differentiated agents does this problem justify?” That is a much more mature question.

The portfolio produced here should be viewed as a pedagogical artifact, not investment advice. The deeper result is the workflow itself: **evidence → specialization → independent judgment → validation → aggregation → deterministic governance → audit**. Once that pattern is understood, the same architecture can be adapted to credit underwriting, M&A screening, scenario analysis, fraud detection, macro surveillance, due diligence, or trading research. The swarm becomes valuable not because any one agent is infallible, but because the system makes many bounded judgments cooperate without surrendering traceability, constraints, or human oversight.
